In [3]:
# pip install selenium webdriver-manager

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

URL = "https://www.hi.co.kr/serviceAction.do?menuId=101210"

def crawl_all_faq():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--blink-settings=imagesEnabled=false")
    options.page_load_strategy = "eager"

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    wait = WebDriverWait(driver, 10)
    driver.get(URL)

    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#faqList")))

    all_rows = []
    seen = set()

    def get_count():
        return len(driver.find_elements(By.CSS_SELECTOR, "#faqList > .title"))

    def click_more_until_end():
        prev = -1
        while True:
            curr = get_count()
            if curr == prev:
                break
            prev = curr

            clicked = False
            btns = driver.find_elements(By.XPATH, "//a[contains(., '더보기')] | //button[contains(., '더보기')]")
            for btn in btns:
                try:
                    if btn.is_displayed() and btn.is_enabled():
                        driver.execute_script("arguments[0].click();", btn)
                        wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "#faqList > .title")) > curr or True)
                        clicked = True
                        break
                except:
                    pass
            if not clicked:
                break

    def collect_current_tab():
        rows = driver.execute_script("""
            const root = document.querySelector('#faqList');
            if (!root) return [];

            const titles = [...root.querySelectorAll(':scope > .title')];
            const panels = [...root.querySelectorAll(':scope > .panel')];
            const out = [];

            for (let i = 0; i < Math.min(titles.length, panels.length); i++) {
                const t = titles[i];
                const p = panels[i];

                const category = t.querySelector('.mark')?.innerText.trim() || '';
                const question = t.querySelector('.toggle p')?.innerText.trim() || '';
                const answer = (p.innerText || '').replace(/\\s+/g, ' ').trim();

                if (question && answer) {
                    out.push({category, question, answer});
                }
            }
            return out;
        """)
        return rows

    # 탭 후보 전부 찾기
    tab_selectors = [
        ".tab_area a", ".tab_area button",
        ".tab a", ".tab button",
        ".tabs a", ".tabs button",
        "[role='tab']"
    ]

    tabs = []
    for sel in tab_selectors:
        found = driver.find_elements(By.CSS_SELECTOR, sel)
        if found:
            tabs = found
            break

    # 탭이 없으면 현재 화면만 수집
    if not tabs:
        click_more_until_end()
        for row in collect_current_tab():
            key = (row["category"], row["question"], row["answer"])
            if key not in seen:
                seen.add(key)
                all_rows.append(row)
    else:
        tab_count = len(tabs)

        for i in range(tab_count):
            # 매번 다시 찾기
            for sel in tab_selectors:
                current_tabs = driver.find_elements(By.CSS_SELECTOR, sel)
                if current_tabs:
                    break

            if i >= len(current_tabs):
                continue

            tab = current_tabs[i]
            tab_name = (tab.text or "").strip()

            try:
                driver.execute_script("arguments[0].click();", tab)
            except:
                continue

            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#faqList")))
            click_more_until_end()

            rows = collect_current_tab()
            added = 0
            for row in rows:
                key = (row["category"], row["question"], row["answer"])
                if key not in seen:
                    seen.add(key)
                    all_rows.append(row)
                    added += 1

            print(f"[탭 {i+1}/{tab_count}] {tab_name} -> {added}건 추가, 누적 {len(all_rows)}건")

    # 출력
    for idx, row in enumerate(all_rows, 1):
        print(f"\n[{idx}] [{row['category']}]")
        print("Q.", row["question"])
        print("A.", row["answer"])
        print("-" * 100)

    print(f"\n총 {len(all_rows)}건 수집 완료")
    driver.quit()

if __name__ == "__main__":
    crawl_all_faq()

[탭 1/46]  -> 10건 추가, 누적 10건
[탭 2/46]  -> 0건 추가, 누적 10건
[탭 3/46]  -> 0건 추가, 누적 10건
[탭 4/46]  -> 0건 추가, 누적 10건
[탭 5/46]  -> 0건 추가, 누적 10건
[탭 6/46]  -> 0건 추가, 누적 10건
[탭 7/46]  -> 0건 추가, 누적 10건
[탭 8/46]  -> 0건 추가, 누적 10건
[탭 9/46] 전체 -> 0건 추가, 누적 10건
[탭 10/46] 인터넷창구 -> 39건 추가, 누적 49건
[탭 11/46] 보험상품 -> 0건 추가, 누적 49건
[탭 12/46] 대출 -> 0건 추가, 누적 49건
[탭 13/46] 보상서비스 -> 0건 추가, 누적 49건
[탭 14/46] 혜택/이벤트 -> 8건 추가, 누적 57건
[탭 15/46]  -> 9건 추가, 누적 66건
[탭 16/46]  -> 0건 추가, 누적 66건
[탭 17/46]  -> 0건 추가, 누적 66건
[탭 18/46]  -> 0건 추가, 누적 66건
[탭 19/46]  -> 10건 추가, 누적 76건
[탭 20/46]  -> 0건 추가, 누적 76건
[탭 21/46]  -> 10건 추가, 누적 86건
[탭 22/46]  -> 2건 추가, 누적 88건
[탭 23/46]  -> 7건 추가, 누적 95건
[탭 24/46]  -> 10건 추가, 누적 105건
[탭 25/46]  -> 10건 추가, 누적 115건
[탭 26/46]  -> 10건 추가, 누적 125건
[탭 27/46]  -> 8건 추가, 누적 133건
[탭 28/46]  -> 0건 추가, 누적 133건
[탭 29/46]  -> 10건 추가, 누적 143건
[탭 30/46]  -> 0건 추가, 누적 143건
[탭 31/46]  -> 2건 추가, 누적 145건
[탭 32/46]  -> 0건 추가, 누적 145건
[탭 33/46]  -> 2건 추가, 누적 147건
[탭 34/46]  -> 9건 추가, 누적 156건
[탭 35/46]  -> 